In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv("../data/processed/featured_data.csv",
                 parse_dates=["StartDatetime"])
df = df.sort_values("StartDatetime").reset_index(drop=True)

# no leakage: Duration, End*, Demand, HighDemand are NOT features
cat_cols = ["ChargerCompany", "ChargerType", "Location", "TimeOfDay"]
num_cols = ["Hour", "Day", "Month", "DayOfWeek", "IsWeekend",
            "User_PrevSessions", "User_AvgPastDemand",
            "Charger_PrevSessions", "Charger_AvgPastDemand", "IsNewUser"]

X = df[cat_cols + num_cols]
y = df["Demand"]

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print(X_train.shape, X_test.shape)

(58274, 14) (14569, 14)


In [3]:
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=42, verbose=-1),
}

results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    mse = mean_squared_error(y_test, pred)
    results.append({"Model": name,
                    "MAE": mean_absolute_error(y_test, pred),
                    "MSE": mse,
                    "RMSE": np.sqrt(mse),
                    "R2": r2_score(y_test, pred)})
    fitted[name] = pipe
    print("done:", name)

results_df = pd.DataFrame(results).sort_values("RMSE").round(4)
results_df

done: Linear Regression
done: Decision Tree
done: Random Forest
done: Gradient Boosting
done: XGBoost
done: LightGBM


,Model,MAE,MSE,RMSE,R2
5,LightGBM,8.2082,125.9225,11.2215,0.3496
2,Random Forest,8.4400,130.1697,11.4092,0.3277
4,XGBoost,8.2964,130.3838,11.4186,0.3266
0,Linear Regression,8.5510,133.6735,11.5617,0.3096
3,Gradient Boosting,9.2284,145.6440,12.0683,0.2478
1,Decision Tree,8.9138,155.3163,12.4626,0.1978


In [5]:
import os, joblib
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

results_df.to_csv("../results/regression_results.csv", index=False)
best = results_df.iloc[0]["Model"]
joblib.dump(fitted[best], "../models/best_regression_model.pkl")
print("Best:", best)

Best: LightGBM
